# VirtuBox Infotech - Data Analyst Assessment
### End-to-End Enterprise Commercial Data Processing, EDA & Statistical Hypothesis Testing
**Dataset:** Global Superstore (51,290 rows × 24 columns)  
**Business Domain:** B2B & Commercial Retail Technology  
**Author:** Candidate (VirtuBox Assessment)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configure visual style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

print("Environment configured successfully.")


## 1. Data Ingestion & Quality Inspection

In [ ]:
# Load raw dataset
df_raw = pd.read_csv('Global_Superstore_Raw.csv', encoding='latin1')
print(f"Total Rows: {df_raw.shape[0]:,}")
print(f"Total Columns: {df_raw.shape[1]}")
print("\nMissing Values:")
print(df_raw.isnull().sum()[df_raw.isnull().sum() > 0])


## 2. Data Cleaning & Feature Engineering

In [ ]:
df = df_raw.copy()

# Standardize dates
df['Order Date'] = pd.to_datetime(df['Order Date'], format='%d-%m-%Y')
df['Ship Date'] = pd.to_datetime(df['Ship Date'], format='%d-%m-%Y')

# Derived time features
df['Order_Year'] = df['Order Date'].dt.year
df['Order_Month'] = df['Order Date'].dt.month
df['Shipping_Days'] = (df['Ship Date'] - df['Order Date']).dt.days

# Impute non-US postal codes
df['Postal Code'] = df['Postal Code'].fillna('Not Recorded')

# Derived metrics
df['Unit_Price'] = np.round(df['Sales'] / df['Quantity'], 2)
df['Unit_Cost'] = np.round((df['Sales'] - df['Profit']) / df['Quantity'], 2)
df['Profit_Margin_%'] = np.round((df['Profit'] / df['Sales']) * 100, 2)
df['Is_Profitable'] = (df['Profit'] > 0).astype(int)

# Discount bands
bins = [-0.001, 0, 0.20, 0.50, 1.0]
labels = ['0% (No Discount)', '1-20% (Low)', '21-50% (Moderate)', '>50% (Heavy)']
df['Discount_Band'] = pd.cut(df['Discount'], bins=bins, labels=labels)

print("Data processing complete. Sample records:")
df[['Order ID', 'Order Date', 'Sales', 'Profit', 'Profit_Margin_%', 'Discount_Band']].head()


## 3. Exploratory Analysis & Hypothesis Testing

### Hypothesis 1: Aggressive Discounting (>20%) Destroys Dollar Profitability

In [ ]:
disc_analysis = df.groupby('Discount_Band', observed=False).agg(
    Order_Count=('Row ID', 'count'),
    Total_Sales=('Sales', 'sum'),
    Total_Profit=('Profit', 'sum'),
    Avg_Profit_Per_Order=('Profit', 'mean')
).reset_index()

disc_analysis['Profit_Margin_%'] = np.round((disc_analysis['Total_Profit'] / disc_analysis['Total_Sales']) * 100, 2)
disc_analysis


In [ ]:
plt.figure(figsize=(9, 5))
colors = ['#2ca02c' if x > 0 else '#d62728' for x in disc_analysis['Profit_Margin_%']]
bars = plt.bar(disc_analysis['Discount_Band'], disc_analysis['Profit_Margin_%'], color=colors, edgecolor='black')
plt.axhline(0, color='black', linewidth=1)
plt.title('Operating Profit Margin (%) by Discount Band', fontsize=14, fontweight='bold')
plt.ylabel('Profit Margin (%)')
for bar in bars:
    y = bar.get_height()
    va = 'bottom' if y >= 0 else 'top'
    plt.text(bar.get_x() + bar.get_width()/2, y + (1.5 if y >= 0 else -6), f'{y:.1f}%', ha='center', va=va, fontweight='bold')
plt.show()


### Hypothesis 2: Segment Efficiency (Corporate B2B vs Consumer)

In [ ]:
seg_analysis = df.groupby('Segment').agg(
    Orders=('Row ID', 'count'),
    Total_Sales=('Sales', 'sum'),
    Total_Profit=('Profit', 'sum'),
    Avg_Order_Value=('Sales', 'mean'),
    Avg_Profit_Order=('Profit', 'mean')
).reset_index()

seg_analysis['Profit_Margin_%'] = np.round((seg_analysis['Total_Profit'] / seg_analysis['Total_Sales']) * 100, 2)
seg_analysis


### 4. Product Sub-Category Loss Centers

In [ ]:
sub_analysis = df.groupby('Sub-Category').agg(
    Total_Sales=('Sales', 'sum'),
    Total_Profit=('Profit', 'sum'),
    Orders=('Row ID', 'count')
).reset_index().sort_values(by='Total_Profit', ascending=True)

sub_analysis['Profit_Margin_%'] = np.round((sub_analysis['Total_Profit'] / sub_analysis['Total_Sales']) * 100, 2)
sub_analysis.head(10)


## 5. Export Clean Processed Data for Google Sheets

In [ ]:
df.to_csv('Processed_Data.csv', index=False)
print("Successfully generated Processed_Data.csv ready for Google Sheets tab 'Processed Data'.")
